# Conservation islands around decoys and supported cryptics

Exploratory analysis of phastCons conservation (track set by `PHASTCONS` in the first cell: 100-way vertebrates or 470-way mammals) at and around the final sites, and
across the introns that hold them, against matched control introns.

- **1.** Introns holding both a decoy and a supported cryptic
- **2.** Controls: same-intron random positions, same-gene length-matched introns, genome-wide
  length- and GC-matched introns (control introns come from the Vast-DB introns with no site)
- **3.** Site-centred metaprofiles, ±100 nt, oriented 5'→3' along the transcript
- **4.** Per-site heatmaps: how many sites sit on a conservation island
- **5.** Intron-level conservation islands away from the site, class vs matched controls

`HNRNPH1_179620582` and its intron (`HsaINT1015786`) are marked throughout.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyBigWig
from scipy import stats

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
PHASTCONS = "470way"   # "100way" (vertebrates) or "470way" (mammals)
FIG = ROOT / "figures" / f"conservation_{PHASTCONS}"
FIG.mkdir(parents=True, exist_ok=True)
BW_PATH = ROOT / "reference" / f"hg38.phastCons{PHASTCONS}.bw"
bw = pyBigWig.open(str(BW_PATH))
CHROM_LEN = bw.chroms()

FLANK = 100
END_TRIM = 100          # intron bases excluded at each end (splice-site conservation)
SMOOTH = 20             # rolling window for island calls
ISLAND_MIN = 0.7        # smoothed phastCons threshold
ISLAND_LEN = 20         # minimum island length (nt)
rng = np.random.default_rng(20260928)

COLOR = {"decoy": "#B2182B", "cryptic_supported": "#2166AC", "same_intron": "#999999",
         "same_gene": "#4D4D4D", "len_gc": "#B8915A"}
LABEL = {"decoy": "Decoys", "cryptic_supported": "Supported cryptics", "same_intron": "Same-intron random position",
         "same_gene": "Same-gene intron, length-matched", "len_gc": "Genome-wide intron, length + GC matched"}

def save(fig, name):
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")

sites = pd.read_csv(ROOT / "results" / "decoy_intron_features_final.tsv", sep="\t",
                    usecols=["decoyID", "site_class", "chr", "start", "strand", "EVENT", "GENE",
                             "intron_start", "intron_end", "intron_length", "n_categories", "psi_count_ge10"])
sites["pos"] = sites["start"] + 1       # 1-based site coordinate
introns = pd.read_csv(ROOT / "results" / "intron_summary_table.tsv", sep="\t",
                      usecols=["GENE", "EVENT", "chr", "intron_start", "intron_end", "strand", "intron_length",
                               "intron_class", "n_decoys", "n_cryptics", "decoy_ids", "cryptic_ids",
                               "intron_gc", "psi_count_ge10"])
HN_SITE = "HNRNPH1_179620582"
HN_EVENT = sites.loc[sites.decoyID == HN_SITE, "EVENT"].iloc[0]
print(sites.site_class.value_counts().to_string()); print(introns.intron_class.value_counts().to_string())
print("HNRNPH1 intron:", HN_EVENT)

## 1. Introns holding both a decoy and a supported cryptic

In [ ]:
both = introns[introns.intron_class == "both"].copy()
both.to_csv(ROOT / "results" / "introns_with_decoy_and_cryptic.tsv", sep="\t", index=False)
print(f"{len(both):,} introns written to results/introns_with_decoy_and_cryptic.tsv")
print(both[["n_decoys", "n_cryptics"]].value_counts().head(8).to_string())
display(both.sort_values("psi_count_ge10", ascending=False).head(10)[
    ["GENE", "EVENT", "chr", "intron_start", "intron_end", "strand", "intron_length", "decoy_ids", "cryptic_ids", "psi_count_ge10"]])

## 2. Matched controls

Every site gets three control positions, all at the same relative distance from the
intron's 5' splice site as the site itself:

- **Same intron**: a random position in the site's own intron, at least 200 nt from the site and
  150 nt from either intron end (controls for intron-wide conservation)
- **Same gene**: the site-free intron of the same gene closest in length (controls for gene-level
  conservation)
- **Genome-wide length + GC**: a site-free intron drawn from the same bin of intron length (50
  quantile bins) and intron GC (10 bins), without replacement (coarsened exact matching)

In [ ]:
pool = introns[introns.intron_class == "none"].copy()
pool["log_len"] = np.log10(pool["intron_length"])
cls_introns = introns[introns.intron_class != "none"].copy()
cls_introns["log_len"] = np.log10(cls_introns["intron_length"])

# Same-gene: nearest site-free intron of the same gene on log length (reuse allowed).
pool_by_gene = {g: d for g, d in pool.groupby("GENE")}
def same_gene_match(row):
    d = pool_by_gene.get(row.GENE)
    if d is None or d.empty:
        return None
    return d.iloc[(d["log_len"] - row.log_len).abs().argmin()]["EVENT"]
cls_introns["ctrl_same_gene"] = [same_gene_match(r) for r in cls_introns.itertuples()]

# Genome-wide coarsened exact matching on length and GC, without replacement.
len_edges = np.quantile(introns["intron_length"], np.linspace(0, 1, 51))
gc_edges = np.quantile(introns["intron_gc"].dropna(), np.linspace(0, 1, 11))
def cem_key(d):
    return (np.clip(np.searchsorted(len_edges, d["intron_length"], side="right") - 1, 0, 49) * 100
            + np.clip(np.searchsorted(gc_edges, d["intron_gc"], side="right") - 1, 0, 9))
pool["cem"] = cem_key(pool); cls_introns["cem"] = cem_key(cls_introns)
avail = {k: list(rng.permutation(d["EVENT"].to_numpy())) for k, d in pool.groupby("cem")}
ctrl_len_gc = []
for k in cls_introns["cem"]:
    lst = avail.get(k, [])
    ctrl_len_gc.append(lst.pop() if lst else None)
cls_introns["ctrl_len_gc"] = ctrl_len_gc

print("introns with a same-gene control:        ", cls_introns.ctrl_same_gene.notna().mean().round(3))
print("introns with a length+GC control:        ", cls_introns.ctrl_len_gc.notna().mean().round(3))
ev = introns.set_index("EVENT")
qc = pd.DataFrame({
    "site introns": cls_introns[["intron_length", "intron_gc"]].median(),
    "same-gene controls": ev.loc[cls_introns.ctrl_same_gene.dropna(), ["intron_length", "intron_gc"]].median(),
    "length+GC controls": ev.loc[cls_introns.ctrl_len_gc.dropna(), ["intron_length", "intron_gc"]].median(),
    "all site-free introns": pool[["intron_length", "intron_gc"]].median(),
})
print("\nMedian length and GC after matching:"); display(qc.round(3))

In [ ]:
# Control positions for every site.
ctrl_map = cls_introns.set_index("EVENT")[["ctrl_same_gene", "ctrl_len_gc"]]
def five_prime(start, end, strand):
    return start if strand == "+" else end
def place(frac, ev_id):
    r = ev.loc[ev_id]
    L = r.intron_length
    off = int(round(frac * (L - 1)))
    return r.intron_start + off if r.strand == "+" else r.intron_end - off, r.chr, r.strand

rows = []
for s in sites.itertuples():
    d = (s.pos - s.intron_start) if s.strand == "+" else (s.intron_end - s.pos)
    frac = d / max(s.intron_length - 1, 1)
    rows.append((s.decoyID, s.site_class, "site", s.chr, s.pos, s.strand, s.EVENT))
    lo, hi = s.intron_start + 150, s.intron_end - 150
    cand = np.arange(lo, hi + 1) if hi > lo else np.array([], dtype=int)
    cand = cand[np.abs(cand - s.pos) >= 200]
    if len(cand):
        rows.append((s.decoyID, s.site_class, "same_intron", s.chr, int(rng.choice(cand)), s.strand, s.EVENT))
    for kind in ("same_gene", "len_gc"):
        cev = ctrl_map.loc[s.EVENT, f"ctrl_{kind}"]
        if isinstance(cev, str):
            p, c, st = place(frac, cev)
            rows.append((s.decoyID, s.site_class, kind, c, int(p), st, cev))
pos = pd.DataFrame(rows, columns=["decoyID", "site_class", "kind", "chr", "pos", "strand", "EVENT"])
print(pos.groupby(["site_class", "kind"]).size().unstack().to_string())

## 3. Site-centred metaprofiles

Per-base phastCons from −100 to +100 nt around each position, oriented along the
transcript (positive = downstream, into the intron for a 5' splice site). Lines are means, bands
95% confidence intervals of the mean.

In [ ]:
def window(chrom, p, strand, flank=FLANK):
    s0, e0 = p - 1 - flank, p + flank
    if s0 < 0 or e0 > CHROM_LEN.get(chrom, 0):
        return np.full(2 * flank + 1, np.nan)
    v = np.asarray(bw.values(chrom, s0, e0, numpy=True), dtype=float)
    return v[::-1] if strand == "-" else v

prof = np.vstack([window(r.chr, r.pos, r.strand) for r in pos.itertuples()])
pos["centre_mean"] = np.nanmean(prof[:, FLANK - 10:FLANK + 11], axis=1)
pos["window_mean"] = np.nanmean(prof, axis=1)
X = np.arange(-FLANK, FLANK + 1)

def mean_ci(M):
    m = np.nanmean(M, axis=0); n = np.sum(~np.isnan(M), axis=0)
    se = np.nanstd(M, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
    return m, m - 1.96 * se, m + 1.96 * se

hn_prof = prof[(pos.decoyID == HN_SITE).to_numpy() & (pos.kind == "site").to_numpy()][0]
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, cls in zip(axes, ["decoy", "cryptic_supported"]):
    for kind in ["site", "same_intron", "same_gene", "len_gc"]:
        sel = ((pos.site_class == cls) & (pos.kind == kind)).to_numpy()
        m, lo, hi = mean_ci(prof[sel])
        key = cls if kind == "site" else kind
        ax.plot(X, m, color=COLOR[key], lw=2.4 if kind == "site" else 1.6, label=f"{LABEL[key]} (n = {sel.sum():,})")
        ax.fill_between(X, lo, hi, color=COLOR[key], alpha=0.2, lw=0)
    ax.axvline(0, color="grey", ls="--", lw=1)
    ax.set_title(LABEL[cls]); ax.set_xlabel("Position relative to site (nt, 5'→3')")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
axes[0].set_ylabel(f"Mean phastCons {PHASTCONS}")
ax2 = axes[0].twinx(); ax2.plot(X, hn_prof, color="black", lw=0.8, alpha=0.6); ax2.set_ylim(0, 1.05)
ax2.set_ylabel("HNRNPH1 decoy (black, right axis)", fontsize=8); ax2.grid(False)
fig.suptitle(f"Conservation around sites vs matched control positions (phastCons {PHASTCONS})", y=1.02)
fig.tight_layout(); save(fig, "1_metaprofile_sites_vs_controls"); plt.show()

summ = pos.groupby(["site_class", "kind"])[["centre_mean", "window_mean"]].mean().round(4)
summ["frac_centre_ge_0.5"] = pos.assign(isl=pos.centre_mean >= 0.5).groupby(["site_class", "kind"])["isl"].mean().round(3)
display(summ)

In [ ]:
# Decoys by support tier, and sites within introns holding both classes
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
ax = axes[0]
tier_col = {1: "#F4A582", 2: "#D6604D", 3: "#67001F"}
site_rows = pos[pos.kind == "site"].merge(sites[["decoyID", "n_categories"]], on="decoyID")
idx_site = pos.index[pos.kind == "site"]
for t in [1, 2, 3]:
    ids = site_rows.loc[(site_rows.site_class == "decoy") & (site_rows.n_categories == t), "decoyID"]
    sel = (pos.kind == "site") & pos.decoyID.isin(ids)
    m, lo, hi = mean_ci(prof[sel.to_numpy()])
    ax.plot(X, m, color=tier_col[t], lw=2, label=f"Decoys, {t} track{'s' if t > 1 else ''} (n = {sel.sum():,})")
    ax.fill_between(X, lo, hi, color=tier_col[t], alpha=0.2, lw=0)
sel = ((pos.kind == "site") & (pos.site_class == "cryptic_supported")).to_numpy()
m, lo, hi = mean_ci(prof[sel]); ax.plot(X, m, color=COLOR["cryptic_supported"], lw=2, label=f"Supported cryptics (n = {sel.sum():,})")
ax.fill_between(X, lo, hi, color=COLOR["cryptic_supported"], alpha=0.2, lw=0)
ax.axvline(0, color="grey", ls="--", lw=1); ax.set_title("By CLIP support tier at the site")
ax.set_xlabel("Position relative to site (nt, 5'→3')"); ax.set_ylabel(f"Mean phastCons {PHASTCONS}"); ax.legend(frameon=False, fontsize=8)

ax = axes[1]
both_events = set(both.EVENT)
for cls in ["decoy", "cryptic_supported"]:
    for kind, ls in [("site", "-"), ("same_gene", ":")]:
        sel = ((pos.site_class == cls) & (pos.kind == kind) &
               pos.decoyID.isin(sites.loc[sites.EVENT.isin(both_events), "decoyID"])).to_numpy()
        m, lo, hi = mean_ci(prof[sel])
        ax.plot(X, m, color=COLOR[cls], ls=ls, lw=2, label=f"{LABEL[cls]}, {'site' if kind == 'site' else 'same-gene control'} (n = {sel.sum():,})")
        ax.fill_between(X, lo, hi, color=COLOR[cls], alpha=0.15, lw=0)
ax.axvline(0, color="grey", ls="--", lw=1); ax.set_title("Introns holding both a decoy and a supported cryptic")
ax.set_xlabel("Position relative to site (nt, 5'→3')"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save(fig, "2_metaprofile_tiers_and_both"); plt.show()

## 4. Which sites sit on a conservation island

Each row is one site's ±100 nt profile, sorted by the mean over the central ±10 nt. The share of
rows with a central mean ≥ 0.5 is in each title.

In [ ]:
panels = [("decoy", "site"), ("cryptic_supported", "site"), ("decoy", "same_gene"), ("cryptic_supported", "same_gene")]
fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=False)
for ax, (cls, kind) in zip(axes, panels):
    sel = ((pos.site_class == cls) & (pos.kind == kind)).to_numpy()
    M = prof[sel]; order = np.argsort(-np.nan_to_num(pos.loc[sel, "centre_mean"].to_numpy()))
    ax.imshow(np.nan_to_num(M[order]), aspect="auto", cmap="magma", vmin=0, vmax=1, interpolation="nearest",
              extent=[-FLANK, FLANK, len(M), 0])
    frac = (pos.loc[sel, "centre_mean"] >= 0.5).mean()
    name = LABEL[cls] if kind == "site" else f"{LABEL[cls]}: same-gene control"
    ax.set_title(f"{name}\n{100 * frac:.1f}% with centre >= 0.5", fontsize=10)
    ax.set_xlabel("nt from position"); ax.axvline(0, color="white", lw=0.5, ls="--")
    if cls == "decoy" and kind == "site":
        r = int(np.where(pos.loc[sel, "decoyID"].to_numpy()[order] == HN_SITE)[0][0])
        ax.annotate("HNRNPH1", (FLANK, r), xytext=(FLANK + 8, r), fontsize=8, fontweight="bold", va="center",
                    arrowprops=dict(arrowstyle="-", lw=0.8), annotation_clip=False)
axes[0].set_ylabel("Sites, sorted by central conservation")
fig.tight_layout(); save(fig, "3_site_heatmaps"); plt.show()

## 5. Conservation islands elsewhere in the intron

Islands are runs of at least 20 nt where the 20-nt rolling mean of phastCons is at least
0.7. The first and last 100 nt of each intron (splice-site conservation) and ±100 nt around every
site (or control position) are excluded, so an island here is conservation *away* from the site —
the "additional conservation islands" of the hnRNPH introns. Each site-bearing intron is compared
with its same-gene and genome-wide matched controls.

In [ ]:
def intron_metrics(ev_id, mask_positions):
    r = ev.loc[ev_id]
    v = np.asarray(bw.values(r.chr, int(r.intron_start) - 1, int(r.intron_end), numpy=True), dtype=float)
    keep = np.ones(len(v), bool); keep[:END_TRIM] = False; keep[-END_TRIM:] = False
    for p in mask_positions:
        i = int(p - r.intron_start)
        keep[max(i - FLANK, 0):min(i + FLANK + 1, len(v))] = False
    body = np.where(keep, np.nan_to_num(v), 0.0)
    sm = np.convolve(body, np.ones(SMOOTH) / SMOOTH, mode="same")
    isl = (sm >= ISLAND_MIN) & keep
    # run lengths of True
    d = np.diff(np.r_[0, isl.astype(int), 0]); starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]
    runs = (ends - starts)[(ends - starts) >= ISLAND_LEN]
    n_body = keep.sum()
    return dict(EVENT=ev_id, body_nt=int(n_body), n_islands=len(runs),
                islands_per_kb=1000 * len(runs) / n_body if n_body else np.nan,
                frac_island=runs.sum() / n_body if n_body else np.nan,
                mean_phastcons=np.nanmean(np.where(keep, v, np.nan)) if n_body else np.nan)

site_pos = pos[pos.kind == "site"].groupby("EVENT")["pos"].apply(list)
recs = []
for r in cls_introns.itertuples():
    m = intron_metrics(r.EVENT, site_pos.get(r.EVENT, [])); m.update(role="site_intron", intron_class=r.intron_class, pair=r.EVENT); recs.append(m)
    for kind in ("same_gene", "len_gc"):
        cev = getattr(r, f"ctrl_{kind}")
        if isinstance(cev, str):
            cp = pos.loc[(pos.kind == kind) & (pos.EVENT == cev), "pos"].tolist()
            m = intron_metrics(cev, cp); m.update(role=kind, intron_class=r.intron_class, pair=r.EVENT); recs.append(m)
isl = pd.DataFrame(recs)
isl.to_csv(ROOT / "results" / f"conservation_island_metrics_{PHASTCONS}.tsv", sep="\t", index=False)
print(f"{len(isl):,} intron rows written to results/conservation_island_metrics_{PHASTCONS}.tsv")
display(isl[isl.EVENT == HN_EVENT])

In [ ]:
ROLE_LABEL = {"site_intron": "Site intron", "same_gene": "Same-gene control", "len_gc": "Length + GC control"}
CLASS_LABEL = {"decoy": "Decoy introns", "cryptic": "Supported-cryptic introns", "both": "Introns with both"}
rows = []
for c in ["decoy", "cryptic", "both"]:
    for role in ["site_intron", "same_gene", "len_gc"]:
        d = isl[(isl.intron_class == c) & (isl.role == role) & (isl.body_nt >= 200)]
        rows.append({"intron_class": c, "role": role, "n": len(d),
                     "any_island": (d.n_islands > 0).mean(), "islands_per_kb": d.islands_per_kb.mean(),
                     "frac_island": d.frac_island.mean(), "mean_phastcons": d.mean_phastcons.mean()})
tab = pd.DataFrame(rows)
display(tab.round(4))

# Paired comparisons: site intron vs its own control (same pair id)
pair_rows = []
for c in ["decoy", "cryptic", "both"]:
    base = isl[(isl.intron_class == c) & (isl.role == "site_intron")].set_index("pair")
    for role in ["same_gene", "len_gc"]:
        ctl = isl[(isl.intron_class == c) & (isl.role == role)].set_index("pair")
        j = base.join(ctl, lsuffix="_site", rsuffix="_ctl", how="inner")
        j = j[(j.body_nt_site >= 200) & (j.body_nt_ctl >= 200)]
        for metric in ["islands_per_kb", "frac_island", "mean_phastcons"]:
            a, b = j[f"{metric}_site"], j[f"{metric}_ctl"]
            w = stats.wilcoxon(a, b, zero_method="zsplit") if len(j) > 10 else None
            pair_rows.append({"intron_class": c, "control": role, "metric": metric, "n_pairs": len(j),
                              "site_mean": a.mean(), "control_mean": b.mean(),
                              "frac_site_higher": (a > b).mean(), "frac_control_higher": (a < b).mean(),
                              "wilcoxon_p": w.pvalue if w else np.nan})
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(FIG / "island_paired_tests.tsv", sep="\t", index=False)
display(pairs.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
metrics = [("any_island", "Fraction of introns with >= 1 island"), ("islands_per_kb", "Islands per kb of intron body"),
           ("mean_phastcons", "Mean phastCons of intron body")]
role_col = {"site_intron": None, "same_gene": "#4D4D4D", "len_gc": "#B8915A"}
cls_col = {"decoy": "#B2182B", "cryptic": "#2166AC", "both": "#762A83"}
w = 0.26
for ax, (m, title) in zip(axes, metrics):
    for i, c in enumerate(["decoy", "cryptic", "both"]):
        for j, role in enumerate(["site_intron", "same_gene", "len_gc"]):
            d = isl[(isl.intron_class == c) & (isl.role == role) & (isl.body_nt >= 200)]
            vals = (d.n_islands > 0).astype(float) if m == "any_island" else d[m]
            mu = vals.mean(); se = vals.std(ddof=1) / np.sqrt(len(vals))
            col = cls_col[c] if role == "site_intron" else role_col[role]
            ax.bar(i + (j - 1) * w, mu, w * 0.9, yerr=1.96 * se, color=col, alpha=0.85 if role == "site_intron" else 0.55,
                   capsize=2, label=ROLE_LABEL[role] if (i == 0 and m == "any_island") else None, edgecolor="white")
    hn = isl[(isl.EVENT == HN_EVENT) & (isl.role == "site_intron")].iloc[0]
    hv = float(hn.n_islands > 0) if m == "any_island" else hn[m]
    ax.scatter([-w], [hv], marker="*", s=200, color="black", zorder=5)
    ax.annotate("HNRNPH1", (-w, hv), xytext=(6, 4), textcoords="offset points", fontsize=8, fontweight="bold")
    ax.set_xticks(range(3), [CLASS_LABEL[c] for c in ["decoy", "cryptic", "both"]], fontsize=9)
    ax.set_title(title, fontsize=10)
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
fig.suptitle(f"Conservation islands away from the site: site introns vs matched controls (phastCons {PHASTCONS}, 95% CI)", y=1.03)
fig.tight_layout(); save(fig, "4_intron_islands_vs_controls"); plt.show()

In [ ]:
# Islands per kb across intron length, to show the comparison is not a length artefact
isl2 = isl.merge(ev[["intron_length"]], left_on="EVENT", right_index=True)
isl2["len_bin"] = pd.qcut(np.log10(isl2.intron_length), 8)
fig, ax = plt.subplots(figsize=(9, 5))
for (c, role), d in isl2[isl2.body_nt >= 200].groupby(["intron_class", "role"]):
    if c == "both":
        continue
    g = d.groupby("len_bin", observed=True)["islands_per_kb"].mean()
    mids = [10 ** iv.mid for iv in g.index]
    col = cls_col[c] if role == "site_intron" else role_col[role]
    ax.plot(mids, g.values, marker="o", color=col, ls="-" if role == "site_intron" else ("--" if role == "same_gene" else ":"),
            lw=2 if role == "site_intron" else 1.4, label=f"{CLASS_LABEL[c]}: {ROLE_LABEL[role]}")
ax.set_xscale("log"); ax.set_xlabel("Intron length (nt, octile bins)"); ax.set_ylabel("Mean islands per kb")
ax.set_title(f"Islands per kb by intron length (phastCons {PHASTCONS})"); ax.legend(frameon=False, fontsize=8)
save(fig, "5_islands_by_length"); plt.show()